# Análisis de Componentes Principales (PCA) exploratorio — TFM Smart Air Madrid

Este notebook documenta y explica el script `pca_exploratorio.py`: qué hace cada parte,
por qué se tomaron ciertas decisiones metodológicas, y cuáles son las conclusiones
principales obtenidas al aplicarlo sobre los datasets normalizados de:

- **Calidad del aire** (`data/normalized/fact/calidad_aire_historico.parquet`)
- **Meteorología AEMET** (`data/normalized/fact/meteo_aemet_historico.parquet`)

> **Nota:** este notebook asume que ya corriste el ETL del proyecto
> (`python src/etl/normalized/main.py --pipeline aire_normalizado` y
> `--pipeline meteo_aemet_normalizado`) y que las tablas `fact/*.parquet` existen.


## 0. Requisitos e imports

Librerías necesarias:
```
pip install pandas pyarrow duckdb scikit-learn matplotlib seaborn
```

- **pandas**: manipulación de tablas
- **duckdb**: motor SQL embebido, usado para agregar y pivotar el detalle horario de
  aire sin agotar la memoria RAM (más rápido y ligero que hacerlo con pandas puro)
- **scikit-learn**: estandarización (`StandardScaler`) y el algoritmo de PCA
- **matplotlib / seaborn**: gráficos (scree plot, heatmap de loadings, proyección)


In [ ]:
from pathlib import Path
import gc

import duckdb
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd()  # ajusta si este notebook no está en la raíz del repo TFM

DATA_FACT_DIR = PROJECT_ROOT / "data" / "normalized" / "fact"
RESULTS_DIR = PROJECT_ROOT / "results" / "pca"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

AIRE_PARQUET = DATA_FACT_DIR / "calidad_aire_historico.parquet"
METEO_PARQUET = DATA_FACT_DIR / "meteo_aemet_historico.parquet"

VARIANZA_OBJETIVO = 0.90  # % de varianza acumulada a alcanzar


## 1. Carga y pivotado de datos

Los datos originales están en **formato largo**: una fila por cada medición individual
(estación, fecha/hora, variable, valor). Para PCA necesitamos **formato ancho**: una fila
por observación (estación + fecha), una columna por variable.

### 1.1 Calidad del aire

Puntos clave de `cargar_y_pivotar_aire`:

1. Los datos son **horarios** (2020-2024, ~24 estaciones × 21 posibles contaminantes) —
   pivotar ese detalle con pandas puro puede agotar la RAM. Por eso se agrega primero a
   **media diaria dentro de DuckDB** (que trabaja con streaming/disco, no todo en memoria),
   reduciendo el volumen ~24 veces antes de que pandas entre en juego.
2. `uom_value` llega como texto (`VARCHAR`) porque los CSV originales usan coma decimal
   (`10,5` en vez de `10.5`) y algunos vienen vacíos — se limpia con
   `REPLACE(...',' , '.')` + `NULLIF` + `TRY_CAST` antes de promediar.
3. Los códigos numéricos de contaminante (`1`, `8`, `14`...) se traducen a nombres legibles
   (`SO2`, `NO2`, `O3`...) haciendo `LEFT JOIN` con `dim_variable_aire.parquet`.


In [ ]:
def cargar_y_pivotar_aire(path: Path) -> pd.DataFrame:
    """Agrega el detalle horario de aire a MEDIA DIARIA + pivota a formato
    ancho, todo dentro de DuckDB (evita agotar la RAM). Traduce códigos de
    magnitud a nombres reales usando dim_variable_aire.parquet.
    """
    dim_path = path.parent.parent / "static_files" / "dim_variable_aire.parquet"
    con = duckdb.connect()

    if dim_path.exists():
        expr_magnitud = "COALESCE(dim.ABREVIATURA, CAST(base.magnitud AS VARCHAR))"
        join_clause = f"""
            LEFT JOIN read_parquet('{dim_path.as_posix()}') AS dim
                ON base.magnitud = dim.MAGNITUD
        """
    else:
        print(f"   No se encontró {dim_path}, se usarán los códigos numéricos.")
        expr_magnitud = "CAST(base.magnitud AS VARCHAR)"
        join_clause = ""

    df_diario_largo = con.execute(f"""
        WITH base AS (
            SELECT
                estacion,
                CAST(fecha AS DATE) AS fecha,
                magnitud,
                TRY_CAST(
                    NULLIF(REPLACE(CAST(uom_value AS VARCHAR), ',', '.'), '')
                    AS DOUBLE
                ) AS valor
            FROM read_parquet('{path.as_posix()}')
        )
        SELECT
            base.estacion, base.fecha, {expr_magnitud} AS magnitud,
            AVG(base.valor) AS valor
        FROM base
        {join_clause}
        GROUP BY base.estacion, base.fecha, {expr_magnitud}
    """).df()
    con.close()

    wide = df_diario_largo.pivot_table(
        index=["estacion", "fecha"], columns="magnitud", values="valor", aggfunc="mean"
    ).reset_index()
    wide.columns.name = None
    return wide


### 1.2 Meteorología

Puntos clave de `cargar_y_pivotar_meteo`:

1. Los datos ya vienen a nivel **diario** (AEMET), así que no hace falta agregación previa
   como en aire.
2. **`viento_direccion` es una variable circular** (grados 0°-360°): 359° y 1° representan
   casi la misma dirección (norte), pero numéricamente están en extremos opuestos de la
   escala. Usarla tal cual distorsiona el PCA (que asume relaciones lineales). Se
   descompone en `viento_direccion_sin` y `viento_direccion_cos`, que sí respetan la
   continuidad circular.


In [ ]:
def cargar_y_pivotar_meteo(path: Path) -> pd.DataFrame:
    """Pivota meteo a formato ancho y descompone viento_direccion (circular,
    en grados) en componentes seno/coseno."""
    df = pd.read_parquet(path)

    wide = df.pivot_table(
        index=["estacion_id", "fecha"], columns="variable_meteo", values="valor",
        aggfunc="mean",
    ).reset_index()
    wide.columns.name = None

    if "viento_direccion" in wide.columns:
        radianes = np.deg2rad(wide["viento_direccion"])
        wide["viento_direccion_sin"] = np.sin(radianes)
        wide["viento_direccion_cos"] = np.cos(radianes)
        wide = wide.drop(columns=["viento_direccion"])

    return wide


## 2. Limpieza: nulos y atípicos

### 2.1 Nulos — ¿por qué eliminar filas en vez de imputar con media/mediana?

Es una decisión metodológica deliberada, no un atajo: **imputar con la media reduce
artificialmente la varianza y distorsiona la matriz de correlaciones**, justo lo que el
PCA usa para construir los componentes. Con volúmenes grandes de datos (decenas de miles
de filas), eliminar las incompletas (*listwise deletion*) es más seguro que sesgar la
estructura de varianza con valores inventados.

Antes de eliminar, se descartan columnas con demasiados nulos (umbral configurable, 30%
por defecto) — esto es clave en calidad del aire, donde no todas las estaciones miden
todos los contaminantes (los metales pesados, por ejemplo, solo se miden en un puñado de
estaciones especializadas).

### 2.2 Atípicos — reportar primero, tratar solo si se justifica

Se usa el método IQR (rango intercuartílico): un valor es atípico si cae fuera de
`[Q1 - 1.5·IQR, Q3 + 1.5·IQR]`. **Por defecto solo se reporta el % de atípicos, no se
modifican** — en aire y meteorología, muchos "atípicos" son eventos reales (una ola de
calor, un episodio de contaminación), no errores de medición. Eliminarlos a ciegas
borraría justo la información más interesante.


In [ ]:
def detectar_atipicos(X: pd.DataFrame, factor_iqr: float = 1.5) -> pd.Series:
    """Detecta atípicos por columna con el método IQR y reporta el % por columna."""
    Q1, Q3 = X.quantile(0.25), X.quantile(0.75)
    IQR = Q3 - Q1
    limite_inf, limite_sup = Q1 - factor_iqr * IQR, Q3 + factor_iqr * IQR
    es_atipico = (X < limite_inf) | (X > limite_sup)
    pct_atipicos = es_atipico.mean().sort_values(ascending=False)
    for col, pct in pct_atipicos.items():
        print(f"   - {col}: {pct:.1%} de valores atípicos")
    return pct_atipicos


def tratar_atipicos_winsorizar(X: pd.DataFrame, factor_iqr: float = 1.5) -> pd.DataFrame:
    """Winsoriza: recorta los atípicos a los límites IQR en vez de eliminar filas."""
    Q1, Q3 = X.quantile(0.25), X.quantile(0.75)
    IQR = Q3 - Q1
    limite_inf, limite_sup = Q1 - factor_iqr * IQR, Q3 + factor_iqr * IQR
    return X.clip(lower=limite_inf, upper=limite_sup, axis=1)


def preparar_matriz_numerica(df_wide, columnas_id, umbral_nulos=0.3,
                              tratar_atipicos=False, factor_iqr=1.5):
    """Descarta columnas con exceso de nulos, elimina filas incompletas,
    y reporta (u opcionalmente trata) atípicos."""
    X = df_wide.drop(columns=columnas_id, errors="ignore")

    prop_nulos = X.isna().mean().sort_values(ascending=False)
    columnas_ok = prop_nulos[prop_nulos <= umbral_nulos].index
    X = X[columnas_ok]

    filas_antes = len(X)
    X = X.dropna()
    print(f"Filas: {filas_antes} -> {len(X)} tras eliminar nulos restantes")

    detectar_atipicos(X, factor_iqr=factor_iqr)
    if tratar_atipicos:
        X = tratar_atipicos_winsorizar(X, factor_iqr=factor_iqr)

    return X


## 3. El PCA en sí

Tres pasos dentro de `ejecutar_pca`:

1. **Estandarización (`StandardScaler`)**: obligatoria antes de PCA porque las variables
   están en escalas distintas (µg/m³, ºC, hPa, %...) — sin esto, las variables con mayor
   magnitud numérica dominarían artificialmente los componentes.
2. **Ajuste del PCA**: calcula los componentes principales y cuánta varianza explica cada
   uno.
3. **Elección del número de componentes**: se acumula la varianza explicada en orden
   hasta alcanzar el umbral (`VARIANZA_OBJETIVO = 0.90` por defecto) — el criterio más
   simple y común, complementario al **criterio del codo** (visual, mirando el scree plot)
   y al **criterio de Kaiser** (autovalor > 1), que se pueden usar para reforzar la
   justificación metodológica en la memoria.


In [ ]:
def ejecutar_pca(X: pd.DataFrame, nombre_dataset: str):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    pca = PCA()
    componentes = pca.fit_transform(X_scaled)

    varianza_exp = pca.explained_variance_ratio_
    varianza_acum = varianza_exp.cumsum()
    n_componentes_objetivo = int((varianza_acum < VARIANZA_OBJETIVO).sum()) + 1

    print(f"=== PCA: {nombre_dataset} ===")
    print(f"Variables: {list(X.columns)}")
    print(f"Varianza explicada: {varianza_exp.round(3)}")
    print(f"Varianza acumulada: {varianza_acum.round(3)}")
    print(f"Componentes para {VARIANZA_OBJETIVO:.0%}: {n_componentes_objetivo}")

    # Scree plot
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(range(1, len(varianza_exp)+1), varianza_acum, marker="o", label="Acumulada")
    ax.bar(range(1, len(varianza_exp)+1), varianza_exp, alpha=0.4, label="Individual")
    ax.axhline(VARIANZA_OBJETIVO, color="red", linestyle="--")
    ax.set_xlabel("Componente principal"); ax.set_ylabel("Varianza explicada")
    ax.set_title(f"Scree plot — {nombre_dataset}"); ax.legend()
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / f"scree_plot_{nombre_dataset}.png", dpi=150)
    plt.show()

    # Loadings
    loadings = pd.DataFrame(
        pca.components_.T,
        columns=[f"PC{i+1}" for i in range(pca.n_components_)],
        index=X.columns,
    )
    loadings.to_csv(RESULTS_DIR / f"loadings_{nombre_dataset}.csv")

    fig, ax = plt.subplots(figsize=(8, max(4, 0.4*len(X.columns))))
    sns.heatmap(loadings.iloc[:, :n_componentes_objetivo], annot=True, fmt=".2f",
                cmap="coolwarm", center=0, ax=ax)
    ax.set_title(f"Loadings (primeros {n_componentes_objetivo} PC) — {nombre_dataset}")
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / f"loadings_heatmap_{nombre_dataset}.png", dpi=150)
    plt.show()

    return pca, loadings, varianza_acum


## 4. Ejecutar el análisis completo

Corre las celdas de abajo para reproducir el PCA sobre tus datos reales.


In [ ]:
# --- Calidad del aire ---
if AIRE_PARQUET.exists():
    df_aire_wide = cargar_y_pivotar_aire(AIRE_PARQUET)
    X_aire = preparar_matriz_numerica(df_aire_wide, columnas_id=["estacion", "fecha"])
    pca_aire, loadings_aire, var_acum_aire = ejecutar_pca(X_aire, "calidad_aire")
    del df_aire_wide
    gc.collect()
else:
    print(f"No se encontró {AIRE_PARQUET}. Corre antes el ETL de aire_normalizado.")


In [ ]:
# --- Meteorología ---
if METEO_PARQUET.exists():
    df_meteo_wide = cargar_y_pivotar_meteo(METEO_PARQUET)
    X_meteo = preparar_matriz_numerica(df_meteo_wide, columnas_id=["estacion_id", "fecha"])
    pca_meteo, loadings_meteo, var_acum_meteo = ejecutar_pca(X_meteo, "meteo")
else:
    print(f"No se encontró {METEO_PARQUET}. Corre antes el ETL de meteo_aemet_normalizado.")


## 5. Conclusiones principales

*(basado en la ejecución real sobre los datos del proyecto)*

### 5.1 Calidad del aire

- De los ~21 códigos de contaminante posibles, **solo 5 tienen cobertura suficiente**
  (<30% de nulos) para el análisis: **CO** (27.0%), **PM10** (17.1%), **SO2** (16.0%),
  **O3** (11.6%) y **NO2** (1.5% — el más completo, con diferencia).
- El resto (NOX, NO, C6H6, PM2.5, y varios compuestos orgánicos volátiles sin nombre
  traducido en el catálogo) se descartó por tener entre 35% y 99.8% de nulos — reflejo de
  que **no todas las estaciones de la red de Madrid miden todos los contaminantes**
  (los compuestos menos comunes solo se miden en estaciones especializadas).
- Tras limpiar nulos quedaron **64,676 observaciones diarias por estación** (de 105,588
  iniciales).
- **La varianza está bastante repartida entre componentes** (36.0% / 19.8% / 18.3% /
  17.6% / 8.3%) — se necesitan **4 de 5 componentes** para llegar al 90% de varianza
  acumulada. Esto indica que **los 5 contaminantes retenidos no están fuertemente
  correlacionados entre sí**: cada uno aporta información bastante independiente
  (coherente con que provienen de fuentes de emisión distintas — tráfico, calefacción,
  fotoquímica atmosférica).
- Se detectaron atípicos moderados (3.3%-5.6% según contaminante) — **no se trataron**,
  ya que probablemente representan episodios reales de contaminación (días sin viento,
  picos de tráfico), información relevante para el análisis, no ruido.

### 5.2 Meteorología

- De 13 variables, solo se descartó **`insolacion`** (36.7% de nulos). Las 12 restantes
  quedaron con buena cobertura (3.9%-28.4% de nulos), resultando en **4,583 observaciones
  diarias completas** (de 8,559 iniciales).
- **La varianza se concentra fuertemente en los primeros componentes**: PC1 explica
  42.3%, PC2 otro 21.6% (63.9% acumulado con solo 2 componentes) — se necesitan
  **5 de 12 componentes** para el 90%. Esto contrasta con el aire: aquí sí hay bastante
  redundancia entre variables (las temperaturas mín/media/máx están altamente
  correlacionadas entre sí, igual que las humedades), por lo que el PCA logra comprimir
  bien la información.
- Se corrigió un problema metodológico importante: **`viento_direccion` es una variable
  circular** (0°-360°) — usarla en grados crudos habría distorsionado el PCA (359° y 1°
  son casi la misma dirección pero numéricamente opuestos). Se descompuso en
  `viento_direccion_sin` / `viento_direccion_cos`.
- `precipitacion` mostró un alto % de "atípicos" (23.3%) por el método IQR, pero esto es
  un artefacto de su naturaleza **zero-inflated** (la mayoría de los días no llueve) —
  no se trató, ya que winsorizar aquí eliminaría precisamente los días de lluvia
  significativa.

### 5.3 Decisiones metodológicas a documentar en la memoria

1. **Nulos**: eliminación de filas (*listwise deletion*) en vez de imputación con
   media/mediana, para no distorsionar la matriz de covarianza que usa el PCA.
2. **Atípicos**: se reportan pero no se eliminan por defecto — se consideran señal
   relevante (eventos ambientales reales), no error de medición.
3. **Estandarización**: obligatoria antes de PCA por la heterogeneidad de escalas y
   unidades entre variables.
4. **Variable circular**: el viento se trató con transformación seno/coseno, no en
   grados crudos.

### 5.4 Próximos pasos sugeridos

- **KMO y test de Bartlett**: para justificar cuantitativamente que el PCA es adecuado
  para estos datos (verificar que las variables están suficientemente correlacionadas).
- **Biplot**: combinar en una sola figura las observaciones (PC1 vs PC2) y los vectores
  de las variables originales, para una interpretación visual más completa.
- **Comunalidades**: cuantificar qué proporción de la varianza de cada variable original
  queda capturada por los componentes retenidos.
